## Импорты и настройки

In [1]:
import pandas as pd
from IPython.display import display, Markdown
from maxidom_hack.calculator import Calculator
from maxidom_hack.db.retriever import ProductRetriever
from maxidom_hack.utils.clients import EmbeddingClient
from maxidom_hack.utils.devices import detect_device
from maxidom_hack.utils.models import Room
from maxidom_hack.utils.parser import ProductParser

# Настройка Pandas: показывать полный текст в ячейках без обрезки (...)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Красивые названия работ
WORK_TITLES = {
    "floor_tile": "Плитка на пол", "wall_tile": "Плитка на стены",
    "floor_laminate": "Ламинат", "wall_paint": "Покраска стен",
    "ceiling_paint": "Покраска потолка", "wall_wallpaper": "Обои",
    "skirting": "Плинтус", "electrics": "Электрика",
}

print("Библиотеки загружены.")

Библиотеки загружены.


## Инициализация компонентов

In [2]:
print("Инициализация моделей...")
device = detect_device()
embed_client = EmbeddingClient(device=device)
retriever = ProductRetriever(embed_client)
calculator = Calculator()
parser = ProductParser()

if retriever.available():
    print(f"[OK] База данных готова! Товаров в базе: {retriever.size:,}")
else:
    print("[ERROR] Ошибка: База данных пуста.")

Инициализация моделей...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

device: cuda:0
dtype: torch.float16
max_seq: 256
Реальное устройство модели: cuda:0
Устройство первого слоя: cuda:0
[OK] База данных готова! Товаров в базе: 49,877


## Тестовые кейсы

In [3]:
test_cases = [
    {
        "name": "1. Санузел (Кап) - Высокая влажность, мрамор",
        "room": Room(length=4, width=3, height=2.7, purpose="bathroom", humidity="high", doors=1, windows=1),
        "variant": "capital",
        "wishes": {"floor_tile": "мрамор", "wall_tile": "светлая керамика", "socket": "черные со шторками"}
    },
    {
        "name": "2. Санузел (Кап) - Низкая влажность, темная плитка",
        "room": Room(length=2, width=3, height=2.7, purpose="bathroom", humidity="low", doors=1, windows=1),
        "variant": "capital",
        "wishes": {"floor_tile": "керамогранит", "wall_tile": "темная плитка"}
    },
    {
        "name": "3. Санузел (Косм) - Влагостойкие обои",
        "room": Room(length=2, width=2, height=2.7, purpose="bathroom", humidity="high", doors=1, windows=0),
        "variant": "cosmetic",
        "wishes": {"wall_wallpaper": "влагостойкие", "wall_paint": "белая"}
    },
    {
        "name": "4. Кухня (Кап) - Фартук и усиленный клей",
        "room": Room(length=3, width=3, height=2.7, purpose="kitchen", humidity="high", doors=1, windows=1),
        "variant": "capital",
        "wishes": {"wall_tile": "фартук", "tile_glue": "усиленный"}
    },
    {
        "name": "5. Кухня (Косм) - Моющаяся краска",
        "room": Room(length=3, width=3, height=2.7, purpose="kitchen", humidity="medium", doors=1, windows=1),
        "variant": "cosmetic",
        "wishes": {"wall_paint": "моющаяся"}
    },
    {
        "name": "6. Гостиная (Кап) - Дуб и серая краска",
        "room": Room(length=5, width=4, height=2.7, purpose="living", humidity="low", doors=1, windows=2),
        "variant": "capital",
        "wishes": {"floor_laminate": "дуб", "wall_paint": "серая"}
    },
    {
        "name": "7. Гостиная (Косм) - Виниловые обои",
        "room": Room(length=5, width=4, height=2.7, purpose="living", humidity="low", doors=1, windows=2),
        "variant": "cosmetic",
        "wishes": {"wall_wallpaper": "виниловые бежевые"}
    },
    {
        "name": "8. Спальня (Косм) - Матовая краска",
        "room": Room(length=3, width=4, height=2.5, purpose="living", humidity="low", doors=1, windows=1),
        "variant": "cosmetic",
        "wishes": {"wall_paint": "матовая", "floor_laminate": "светлый"}
    },
    {
        "name": "9. Коридор (Кап) - Светлая плитка",
        "room": Room(length=1.5, width=4, height=2.7, purpose="hallway", humidity="low", doors=2, windows=0),
        "variant": "capital",
        "wishes": {"floor_tile": "светлая", "cable": "ВВГнг"}
    },
    {
        "name": "10. Стресс-тест (Мини-санузел)",
        "room": Room(length=1, width=1, height=2.5, purpose="bathroom", humidity="high", doors=1, windows=0),
        "variant": "capital",
        "wishes": {}
    }
]

## Запуск тестов

In [4]:
for case in test_cases:
    display(Markdown(f"---\n### ТЕСТ: {case['name']}"))
    print(f"**Параметры:** {case['room'].length}x{case['room'].width}x{case['room'].height} м, Назначение: {case['room'].purpose}, Влажность: {case['room'].humidity}\n")

    picked_products = []
    if case["variant"] == "capital" and case["room"].purpose in ("bathroom", "kitchen"):
        cats = ["floor_tile", "wall_tile", "tile_glue", "tile_grout", "skirting", "socket", "switch", "cable"]
    elif case["variant"] == "capital":
        # В капитальном ремонте стены красят (убираем обои)
        cats = ["floor_laminate", "wall_paint", "skirting", "socket", "switch", "cable"]
    else:
        # В косметике смотрим, что пользователь выбрал в пожеланиях
        if "wall_wallpaper" in case["wishes"]:
            cats = ["floor_laminate", "wall_wallpaper", "skirting"]
        else:
            cats = ["floor_laminate", "wall_paint", "skirting"]
    
    for cat in cats:
        wish = case["wishes"].get(cat, "")
        if wish:
            res = retriever.search_raw(query=wish, category_keys=[cat], limit=3)
            metas = (res.get("metadatas") or [[]])[0]
            docs = (res.get("documents") or [[]])[0]
        else:
            res = retriever.filter(category_keys=[cat], limit=3)
            metas = res.get("metadatas") or []
            docs = res.get("documents") or []
        
        if not metas: continue
        meta, doc = metas[0], docs[0]
        try:
            product = parser.parse_product(meta, doc)
            picked_products.append(product)
        except Exception: pass

    # 1. Таблица товаров
    display(Markdown("#### || Товары, найденные RAG:"))
    products_data = [{
        "Категория": p.category, "Товар": p.name, 
        "Площадь упак.": p.package_area, "Объем упак.": p.quantity_in_package, 
        "Расход": p.consumption_per_m2, "Цена": p.price, "Ед.": p.unit
    } for p in picked_products]
    display(pd.DataFrame(products_data).style.format(precision=2))

    # 2. Таблица сметы
    estimate = calculator.calc_estimate(case["room"], case["variant"], picked_products)
    
    display(Markdown(f"#### ||| Смета (Итого: {estimate.total:,.0f} ₽):"))
    estimate_data = []
    for work in estimate.works:
        if work.items:
            for item in work.items:
                estimate_data.append({
                    "Работа": WORK_TITLES.get(work.work, work.work),
                    "Товар": item.name,
                    "Кол-во": item.qty,
                    "Ед.": item.unit,
                    "Сумма": f"{item.total:,.0f} ₽",
                    "Обоснование (формула)": item.reason
                })
    display(pd.DataFrame(estimate_data).style.set_properties(**{'text-align': 'left'}))
    print("\n")

---
### ТЕСТ: 1. Санузел (Кап) - Высокая влажность, мрамор

**Параметры:** 4x3x2.7 м, Назначение: bathroom, Влажность: high



#### || Товары, найденные RAG:

,Категория,Товар,Площадь упак.,Объем упак.,Расход,Цена,Ед.
0,floor_tile,керамогранит GLOBAL TILE Amore 60x60 белый,1.44,nan,nan,2125.00,уп.
1,wall_tile,"плитка настенная KERLIFE Calacatta Silver New 31,5x63 белый",1.79,nan,nan,2689.00,уп.
2,tile_glue,клей плиточный ОСНОВИТ Гранипликс АС15 R быстротвердеющий водо-морозостойкий 5кг,nan,5.00,2.10,449.00,шт.
3,tile_grout,"затирка для швов PLITONIT Colorit Premium 0,5-13мм 2кг салатовая",nan,2.00,nan,515.00,шт.
4,skirting,"плинтус потолочный СОЛИД 60х58х2000мм белый, арт.С17/60",nan,nan,nan,109.00,шт.
5,socket,розетка встраиваемая Эра с заземлением шторки антрацит,nan,nan,nan,129.00,шт.
6,switch,выключатель 1-клавишный HEGEL Alfa IP44 10А белый,nan,nan,nan,352.00,шт.
7,cable,"кабель NYMнг(А)-LS-О 2x2,5 ГОСТок серый 5 м",nan,5.00,nan,729.00,шт.


#### ||| Смета (Итого: 30,634 ₽):

...

**Параметры:** 1x1x2.5 м, Назначение: bathroom, Влажность: high



#### || Товары, найденные RAG:

,Категория,Товар,Площадь упак.,Объем упак.,Расход,Цена,Ед.
0,floor_tile,"керамогранит CERSANIT Infinity 29,7x59,8 темно-серый рельеф",1.77,nan,nan,2720.00,уп.
1,wall_tile,плитка настенная ALMA CERAMICA Style 20х60 серый 707,1.68,nan,nan,1877.00,уп.
2,tile_glue,клей плиточный ОСНОВИТ Гранипликс АС15 R быстротвердеющий водо-морозостойкий 5кг,nan,5.00,2.10,449.00,шт.
3,tile_grout,"затирка для швов PLITONIT Colorit Premium 0,5-13мм 2кг салатовая",nan,2.00,nan,515.00,шт.
4,skirting,"плинтус потолочный СОЛИД 60х58х2000мм белый, арт.С17/60",nan,nan,nan,109.00,шт.
5,socket,розетка 1 пост ONEKEYELECTRO Rotondo с/з черный,nan,nan,nan,503.00,шт.
6,switch,выключатель 1-клавишный HEGEL Alfa IP44 10А белый,nan,nan,nan,352.00,шт.
7,cable,"кабель NYMнг(А)-LS-О 2x2,5 ГОСТок серый 5 м",nan,5.00,nan,729.00,шт.


#### ||| Смета (Итого: 6,718 ₽):

,Работа,Товар,Кол-во,Ед.,Сумма,Обоснование (формула)
0,Плитка на пол,"керамогранит CERSANIT Infinity 29,7x59,8 темно-серый рельеф",1,уп.,"2,720 ₽",Площадь 1 м2 + запас 5% (0.05 м2) / упаковка 1.77 м2 = 1 шт. упак.
1,Плитка на пол,клей плиточный ОСНОВИТ Гранипликс АС15 R быстротвердеющий водо-морозостойкий 5кг,1,шт.,449 ₽,Площадь 1 м² × норма 2.1 кг/м² ÷ 5.0 кг (мешок) = 1 шт.
2,Плитка на пол,"затирка для швов PLITONIT Colorit Premium 0,5-13мм 2кг салатовая",1,шт.,515 ₽,Площадь 1 м² × норма 1.5 кг/м² ÷ 2.0 кг = 1 шт.
3,Плинтус,"плинтус потолочный СОЛИД 60х58х2000мм белый, арт.С17/60",2,шт.,218 ₽,Периметр 4 м - 1 двер(и) × 0.9 м = 3.1 м. Планок по 2.5 м: 2 шт.
4,Электрика,розетка 1 пост ONEKEYELECTRO Rotondo с/з черный,2,шт.,"1,006 ₽",Эвристика: 1 розетка на 6 м² (1 м² / 6 = 2 шт.)
5,Электрика,выключатель 1-клавишный HEGEL Alfa IP44 10А белый,1,шт.,352 ₽,Эвристика: по 1 выключателю на вход
6,Электрика,"кабель NYMнг(А)-LS-О 2x2,5 ГОСТок серый 5 м",2,шт.,"1,458 ₽",Эвристика: 1.5 м кабеля на 1 м² пола. Нужно 10 м. В упаковке 5 м. Итого: 2 шт.
